In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, log_loss, classification_report
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import export_graphviz
import graphviz
import pydotplus
from IPython.display import Image

In [3]:
file_path = input("Please enter the file path: ")
df = pd.read_csv(file_path, delimiter=',', encoding='latin1')
# Encoding 'latin1' is used to ensure the ë can be read by Pandas


In [ ]:
df.head(20)

## Prep/remap

In [ ]:
y = df['nogmdiag']
X = df.drop(columns=['nogmdiag', 'act_v11b', 'act_v11d', 'tumsoort'])  #took out minutes in wk and wknd

# summed act_v11a and act_v11c into their own named column, 'sed_beh' then dropped them
X['sed_beh'] = X['act_v11a'] + X['act_v11c']
X = X.drop(columns=['act_v11a', 'act_v11c'])
X['sed_beh'] = X['sed_beh'].round(2)

y = y.replace({1: 0, 4: 0, 2: 1, 3: 1}) #dichotimized y train
print(y.value_counts())

# Rename columns
X.rename(columns={
    'leeft': 'age',
    'alg_v1b': 'sex',
    'zorg_v6': 'follow_up_status',
    'stijl_v1a': 'smoking',
    'stijl_v3a': 'alcohol',
    'stijl_v5': 'drug_use',
    'fam_v1':'family_history',
    'ther_chemo': 'chemotherapy',
    'ther_rt': 'radiation',
    'ther_ht': 'hormone_therapy',
}, inplace=True)

X['smoking'] = X['smoking'].replace({1: 0, 2: 0, 3: 1})
X['alcohol'] = X['alcohol'].replace({1: 0, 2: 0, 3: 1})
X['follow_up_status'] = X['follow_up_status'].replace({2: 0})
X['family_history'] = X['family_history'].replace({1: 0, 2: 1, 3: 1, 4: 2}) #2 because they kept 'idk'.

# Print the first few rows to verify the mappings
print(X.head(20))

In [ ]:
print(X['drug_use'].value_counts())

## Split age into: [0: 18-24] [1: 25-34] [2: 35-39]

In [ ]:
# Function to split the 'age' column into categories
def map_age(age):
    if 18 <= age <= 24:
        return 0
    elif 25 <= age <= 34:
        return 1
    elif 35 <= age <= 39:
        return 2
    else:
        return None

# Apply the function to the 'age' column
X['age'] = X['age'].apply(map_age).astype('float64')

# Display the DataFrame
print(X)
print(X.head)

## Median (IQR)

In [ ]:
# median (IQR) of sedentary behaviour for total pop
# Calculate the median
median_value_X = X['sed_beh'].median()
print(f'Median: {median_value_X:.2f}')

# Calculate the 25th percentile (Q1)
q1 = X['sed_beh'].quantile(0.25)
print(f'25th percentile (Q1): {q1:.2f}')

# Calculate the 75th percentile (Q3)
q3 = X['sed_beh'].quantile(0.75)
print(f'75th percentile (Q3): {q3:.2f}')

# Calculate the IQR
iqr = q3 - q1
print(f'IQR: {iqr:.2f}')

In [9]:
# 1 if sed_beh is over median (for now)
X['sed_beh'] = X['sed_beh'].apply(lambda x: 1 if x >= 14.29 else 0).astype('float64') 

In [ ]:
print (X.head(20))

In [ ]:
X.info()

In [ ]:
X.hist(figsize=(15, 15))

## Introduce train/test & Scale the data

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize/scale the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## AIC, BIC

In [ ]:
from sklearn.linear_model import LassoLarsIC

# Fit LassoLars with AIC criterion
model_aic = LassoLarsIC(criterion='aic')
model_aic.fit(X_train, y_train)

# Fit LassoLars with BIC criterion
model_bic = LassoLarsIC(criterion='bic')
model_bic.fit(X_train, y_train)

# Plot the results
alphas_aic = model_aic.alphas_
alphas_bic = model_bic.alphas_

plt.plot(alphas_aic, model_aic.criterion_, label="AIC")
plt.plot(alphas_bic, model_bic.criterion_, label="BIC")
plt.xlabel('Alpha')
plt.ylabel('Criterion Value')
plt.legend()
plt.title('Lasso Model Selection with AIC and BIC')
plt.show()

In [ ]:
print(y_test.shape)
print(y_train.shape)
print(X_test.shape)
print(X_train.shape)

## Check correlation with cancer recurrence

In [ ]:
df.corr()['nogmdiag'].sort_values(ascending=False)

## L1 Ratio - Logistic Regression

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train) # calculate weights of each class in a balanced manner
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Define the parameter grid for the log reg
param_grid = {
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9, 1.0],  # Range of l1_ratio values to search
    'C': [0.1, 1, 10, 100]  # Range of regularization strengths
    # right now c = 10 but is thgat still good enough?
}

# Initialize the logistic regression model with class weights
elastic_model = LogisticRegression(max_iter=1000, class_weight=class_weight_dict)

# Initialize GridSearchCV
grid_search = GridSearchCV(elastic_model, param_grid, cv=5, scoring='accuracy')

# Fit the model using the training data
grid_search.fit(X_train, y_train)

# Retrieve the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Use the best parameters to initialize the final model
elastic_model = LogisticRegression(**best_params, class_weight=class_weight_dict).fit(X_train, y_train)

In [20]:
# Make predictions using the testing set
y_elas_pred = elastic_model.predict(X_test) #predicted labels for X test set. Returns class w highest prob for each sample. Binary outcome
y_pred_prob = elastic_model.predict_proba(X_test)[:, 1] #predicted probabilities of the positive class (class 1)

## Evaluations

In [ ]:
#Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_elas_pred)
plt.figure(figsize=(4, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Predicted Negative', 'Predicted Positive'],
            yticklabels=['Actual Negative', 'Actual Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# Correlation matrix
correlation_matrix = X.corr()
plt.figure(figsize=(8, 8))
sns.set(font_scale=0.7)
sns.heatmap(correlation_matrix, annot=True, cmap="YlGn", linewidths=.5)
plt.title("Correlation Matrix Heatmap of All Variables")
plt.legend(loc='best')
plt.show()

print('Classification Report:')
print(classification_report(y_test, y_elas_pred))

# The coefficients
coef = elastic_model.coef_

# Convert coefficients to odds ratios
odds_ratios = np.exp(coef)

# ROC Curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob, average='micro') # micro: Calculate metrics globally by counting the total true positives, false negatives and false positives.    

plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='best')
plt.show()
print('AUC:', auc)

unique_test, counts_test = np.unique(y_test, return_counts=True)
print(f'Class distribution in y_pred: {dict(zip(unique_test, counts_test))}')

# Flatten the arrays to ensure they are 1-dimensional
coef_odds = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': coef.flatten(),
    'Odds Ratio': odds_ratios.flatten()
})

coef_odds_sorted = coef_odds.sort_values(by='Coefficient', ascending=False)
print(coef_odds_sorted)



## Decision tree - feature importances (entropy)

In [ ]:
# Train a decision tree to calculate feature importances (entropy)
tree_model = DecisionTreeClassifier(criterion='entropy', random_state=42) #gini better
tree_model.fit(X_train, y_train)

# Make predictions using the testing set
y_tree_pred = tree_model.predict(X_test)  # Predicted labels for X_test
y_tree_pred_prob = tree_model.predict_proba(X_test)[:, 1]  # Predicted probabilities of the positive class (class 1)

# Calculate training accuracy
train_accuracy_tree = accuracy_score(y_test, y_tree_pred)
print(f'Training Accuracy Tree: {train_accuracy_tree:.2f}')

print('Classification Report Tree:')
print(classification_report(y_test, y_tree_pred))

# Print feature importances
feature_importances_tree = tree_model.feature_importances_
print("Feature Importances:", feature_importances_tree)

# Create a DataFrame to map feature importances to column names
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances_tree
})

# Sort the DataFrame by importance
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Display the feature importances
print(feature_importance_df)

# Select features with entropy above the cutoff
entropy_cutoff = 0.8
selected_features = feature_importances_tree > entropy_cutoff
print(f"Selected features: {selected_features}")

# Get the names of the selected features
top_features = np.arange(X.shape[1])[selected_features]
print(f"Top features: {top_features}")

# Export the decision tree to a DOT file
dot_data = export_graphviz(tree_model, out_file=None, 
                        feature_names=X.columns,
                        class_names=['Class 0', 'Class 1'],
                        filled=True, rounded=True, special_characters=True)

# Use graphviz to create a graph from the DOT data
graph = pydotplus.graph_from_dot_data(dot_data)

# Display the graph
display(Image(graph.create_png()))